# vLLM Deployment Tests

Run these tests in order. Start by creating and validating an AWS GPU instance, then move through each vLLM deployment phase from the README.

AWS GPU Instance Setup 

- Launch an EC2 GPU instance such as `g5.xlarge`.
- Use Ubuntu Deep Learning AMI or Ubuntu 22.04 with NVIDIA drivers.
- Allow SSH from your IP.
- Allow TCP `8000` from your IP for API testing.
- SSH into the instance.
- Confirm the GPU with `nvidia-smi`.

# Requests

In [12]:
import requests
import json

IPADDRESS = "43.204.103.137"

url = f"http://{IPADDRESS}:8000/v1/chat/completions -H Authorization: Bearer affgt6tvhhjjer74"  # adjust host/port

payload = {
    "model": "Qwen/Qwen3-0.6B",  # must match what vLLM was launched with
    "messages": [
        {"role": "user", "content": "Explain RAG in one sentence."}
    ],
    "max_tokens": 500,
    "temperature": 0.1
}

response = requests.post(url, json=payload)
response.raise_for_status()

data = response.json()
print(data["choices"][0]["message"]["content"])
print("Tokens used:", data["usage"])

HTTPError: 401 Client Error: Unauthorized for url: http://43.204.103.137:8000/v1/chat/completions%20-H%20Authorization:%20Bearer%20affgt6tvhhjjer74

# vllm

In [2]:
from vllm import LLM, SamplingParams

llm = LLM(model="your-model-name", gpu_memory_utilization=0.9)

sampling_params = SamplingParams(temperature=0.7, max_tokens=200)

prompts = ["Explain RAG in one sentence."]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(output.outputs[0].text)

ModuleNotFoundError: No module named 'vllm'

# Openai

In [13]:
from openai import OpenAI

client = OpenAI(
    base_url=f"http://{IPADDRESS}:8000/v1",
    api_key="affgt6tvhhjjer74r"  # vLLM doesn't enforce auth by default
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[
        {"role": "user", "content": "Explain RAG in one sentence."}
    ],
    max_tokens=2000,
    temperature=0.2,
)

print(response.choices[0].message.content)
print("Usage:", response.usage)

<think>
Okay, the user wants a one-sentence explanation of RAG. Let me start by recalling what RAG stands for. RAG is Retrieval-Augmented Generation, right? So it's a method that combines retrieval and generation.

I need to make sure the sentence is concise but covers both components. Maybe start with "Retrieval-Augmented Generation" as the main term. Then explain that it uses a large pre-trained model to generate text based on a retrieved context. 

Wait, should I mention the specific components? Like the model, the retrieval step, and the generation step? But the user asked for one sentence. So maybe keep it simple. 

Also, check if there's any technical jargon that's necessary. Since it's a common term, it should be understandable. Let me structure it: "Retrieval-Augmented Generation (RAG) is a method that combines a large pre-trained language model with a retrieval step to generate contextually relevant text." 

That seems to cover both the retrieval and generation parts. Let me v